# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access the dataset metadata as an object and print its main attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.date_published}\nVersion: {dataset.metadata.version}\nIdentifier: {dataset.metadata.identifier}\nLicense: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs. We'll use the Croissant metadata to print the available record sets and their fields, always referencing them by their `@id`.

In [ ]:
# List all record sets and their field IDs

record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- Record Set: name={rs.name}, @id={rs.id}")
    fields = rs.fields
    print(f"  Fields ({len(fields)}):")
    for field in fields:
        print(f"    - {field.name} (@id={field.id}, dataType={field.data_type})")
    print('')

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, using the record set and field `@id`s from the overview.

We'll create a pandas DataFrame for each record set using their `@id` and display an overview of the columns (by `@id`) for one example record set.

In [ ]:
# Extract data from each record set using their @id

dataframes = {}
for rs in dataset.record_sets:
    recs = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(recs)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} records for RecordSet {rs.name} (@id={rs.id}) with columns: {df.columns.tolist()}")
    print(df.head(1))
    print('---')

# Choose the first record set for demonstration in following steps
if len(dataset.record_sets) > 0:
    main_record_set = dataset.record_sets[0]
    main_rs_id = main_record_set.id
    print(f"Selected main record set: {main_record_set.name} (@id={main_rs_id})")
else:
    main_record_set = None
    main_rs_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will select a numeric field (column) by its `@id` from the main record set. We'll perform filtering, normalization, and grouping operations.

In [ ]:
import numpy as np

if main_rs_id is not None and len(dataframes[main_rs_id]) > 0:
    df = dataframes[main_rs_id]
    # List all numeric fields from the record set (by Croissant dataType "Integer" or "Float")
    numeric_fields = [field for field in main_record_set.fields if field.data_type in ['Integer','Float','Number']]
    if numeric_fields:
        # Pick first numeric field for demonstration
        numeric_field = numeric_fields[0].id
        print(f"Selected numeric field @id: {numeric_field} (name: {numeric_fields[0].name})")
    else:
        numeric_field = df.select_dtypes(include=[np.number]).columns[0] if len(df.select_dtypes(include=[np.number]).columns) > 0 else None
        print(f"Auto-selected numeric field (by dtype): {numeric_field}")
    # Filtering
    if numeric_field and numeric_field in df.columns:
        threshold = df[numeric_field].mean()  # example: values above the mean
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): found {len(filtered_df)} records.")
        print(filtered_df.head())
        # Normalization (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records (mean=0, std=1):")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group operation - group by the first non-numeric field
        group_fields = [field.id for field in main_record_set.fields if field.data_type not in ['Integer','Float','Number']]
        group_field = group_fields[0] if group_fields and group_fields[0] in df.columns else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numerical columns to filter or normalize.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we'll create a histogram and boxplot for our selected numeric field, and a bar plot if a group field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id is not None and (numeric_field is not None) and (numeric_field in dataframes[main_rs_id].columns):
    df = dataframes[main_rs_id]
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')

    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field].dropna())
    plt.title(f'Boxplot of {numeric_field}')
    plt.tight_layout()
    plt.show()
    # Group plot if available
    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(8,4))
        order = df[group_field].value_counts().index[:10] # Top 10 groups
        sns.barplot(
            data=df[df[group_field].isin(order)],
            y=group_field,
            x=numeric_field,
            order=order,
            ci=None,
        )
        plt.title(f'Mean {numeric_field} by {group_field} (Top 10)')
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to load and explore the FAIR² mass spectrometry dataset on extracellular vesicles from stimulated human mast cells.

- We loaded Croissant metadata and all available record sets, inspecting structure and using field `@id`s throughout for rigorous referencing.
- We extracted records to pandas DataFrames and highlighted how to filter, normalize, and group data by fields identified by their `@id`s rather than just column names.
- Basic visualizations of key numeric fields (by their `@id`) were provided for exploratory analysis.

Further analyses can combine other record sets, advanced visualizations, or downstream ML workflows as needed using this reproducible structure.